# Task 1: Model Training and Optimization Pipeline
Use this notebook to perform your data preprocessing, hyperparameter tuning via Cross-Validation, and final evaluation on the test set.

In [1]:
import pandas as pd
import numpy as np
import pickle
import time
import optuna
import trackio
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, cross_val_score
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import LabelEncoder

# Add any other imports you need here

## 1. Data Loading & Preprocessing
Load `train.csv` and `test.csv`. Convert string categorical variables to numeric.
**Required:** Save your label encoders/mappings because your Streamlit UI will need them later to prepare user inputs for inference!

In [3]:
train_df = pd.read_csv('Dataset/train.csv')
test_df = pd.read_csv('Dataset/test.csv')

# TODO: Implement your preprocessing here (use LabelEncoder or manual dictionaries)
# Ensure you keep all necessary features that will be shown on the UI dashboard.
location_encoder = LabelEncoder()
city_encoder = LabelEncoder()
status_encoder = LabelEncoder()
property_type_encoder = LabelEncoder()

categorical_columns = {
    'location': location_encoder,
    'city': city_encoder,
    'Status': status_encoder,
    'property_type': property_type_encoder,
}

for column_name, encoder in categorical_columns.items():
    train_df[column_name] = train_df[column_name].astype(str)
    test_df[column_name] = test_df[column_name].astype(str)

    train_df[column_name] = encoder.fit_transform(train_df[column_name])
    mapping = {label: index for index, label in enumerate(encoder.classes_)}
    test_df[column_name] = test_df[column_name].map(mapping).fillna(-1).astype(int)

feature_columns = [column for column in train_df.columns if column not in ['price']]

# Separate predictors (X) and target (y: 'price')
X_train = train_df[feature_columns]
y_train = train_df['price']

X_test = test_df[feature_columns]
y_test = test_df['price']

## 2. Hyperparameter Tuning using Cross-Validation

**Strict Search Space:**
- `n_estimators`: 50 to 200
- `max_depth`: 10 to 30
- `min_samples_split`: 2 to 10

Implement Grid Search, Random Search, and Bayesian Optimization (using Optuna). Evaluate each using 5-fold cross-validation on `train_df`.

In [ ]:
rf = RandomForestRegressor(random_state=42, n_jobs=-1)

# TODO: Initialize trackio project/experiment here
trackio.init(project='STTAI_Assignment4', name='v4.0', config={'model': 'RandomForestRegressor'})

# TODO: 1. Grid Search Implementation
# Use trackio to log the method name, time taken, number of iterations, and best cross-validation score
param_grid = {
    'n_estimators': [50, 100, 150, 200],
    'max_depth': [10, 15, 20, 25, 30],
    'min_samples_split': [2, 5, 8],
}
grid_search = GridSearchCV(estimator=rf, param_grid=param_grid, cv=5, n_jobs=-1, verbose=2, scoring='neg_mean_absolute_error')
start_time = time.time()
grid_search.fit(X_train, y_train)
end_time = time.time()
grid_time_taken = end_time - start_time
grid_best_mae = -grid_search.best_score_

trackio.log({
    'method': 'Grid Search',
    'grid_time_taken': grid_time_taken,
    'grid_num_iterations': len(grid_search.cv_results_['params']),
    'grid_best_mae': float(grid_best_mae),
    'grid_n_estimators': grid_search.best_params_['n_estimators'],
    'grid_max_depth': grid_search.best_params_['max_depth'],
    'grid_min_samples_split': grid_search.best_params_['min_samples_split'],
})

# TODO: 2. Random Search Implementation
# Use trackio to log the method name, time taken, number of iterations, and best cross-validation score
param_dist = {
    'n_estimators': range(50, 201),
    'max_depth': range(10, 31),
    'min_samples_split': range(2, 11),
}
random_search = RandomizedSearchCV(estimator=rf, param_distributions=param_dist, n_iter=60, cv=5, n_jobs=-1, verbose=2, random_state=42, scoring='neg_mean_absolute_error')
start_time = time.time()
random_search.fit(X_train, y_train)
end_time = time.time()
random_time_taken = end_time - start_time
random_best_mae = -random_search.best_score_

trackio.log({
    'method': 'Random Search',
    'random_time_taken': random_time_taken,
    'random_num_iterations': 60,
    'random_best_mae': float(random_best_mae),
    'random_n_estimators': random_search.best_params_['n_estimators'],
    'random_max_depth': random_search.best_params_['max_depth'],
    'random_min_samples_split': random_search.best_params_['min_samples_split'],
})


# TODO: 3. Bayesian Optimization (Optuna) Implementation
# Use trackio to log the method name, time taken, number of iterations, and best cross-validation score
def objective(trial):
    n_estimators = trial.suggest_int('n_estimators', 50, 200)
    max_depth = trial.suggest_int('max_depth', 10, 30)
    min_samples_split = trial.suggest_int('min_samples_split', 2, 10)

    rf = RandomForestRegressor(n_estimators=n_estimators, max_depth=max_depth, min_samples_split=min_samples_split, random_state=42, n_jobs=-1)
    score = cross_val_score(rf, X_train, y_train, cv=5, scoring='neg_mean_absolute_error').mean()
    return float(score)
study = optuna.create_study(direction='maximize')
start_time = time.time()
study.optimize(objective, n_trials=60)
end_time = time.time()
bayes_time_taken = end_time - start_time
bayes_best_mae = -study.best_value

trackio.log({
    'method': 'Bayesian Optimization',
    'bayes_time_taken': bayes_time_taken,
    'bayes_num_iterations': 60,
    'bayes_best_mae': float(bayes_best_mae),
    'bayes_n_estimators': study.best_params['n_estimators'],
    'bayes_max_depth': study.best_params['max_depth'],
    'bayes_min_samples_split': study.best_params['min_samples_split'],
})

* Created new run: v4.0
Fitting 5 folds for each of 60 candidates, totalling 300 fits


[I 2026-04-14 12:52:41,093] A new study created in memory with name: no-name-bae753f3-6efd-4299-8c52-d56f5df5345e
[I 2026-04-14 12:52:52,134] Trial 0 finished with value: -13621.577481120632 and parameters: {'n_estimators': 92, 'max_depth': 17, 'min_samples_split': 4}. Best is trial 0 with value: -13621.577481120632.
[I 2026-04-14 12:53:02,257] Trial 1 finished with value: -13580.067471864098 and parameters: {'n_estimators': 92, 'max_depth': 26, 'min_samples_split': 5}. Best is trial 1 with value: -13580.067471864098.
[I 2026-04-14 12:53:14,887] Trial 2 finished with value: -13776.16813278157 and parameters: {'n_estimators': 145, 'max_depth': 19, 'min_samples_split': 7}. Best is trial 1 with value: -13580.067471864098.
[I 2026-04-14 12:53:27,162] Trial 3 finished with value: -13448.372935700003 and parameters: {'n_estimators': 123, 'max_depth': 20, 'min_samples_split': 3}. Best is trial 3 with value: -13448.372935700003.
[I 2026-04-14 12:53:41,956] Trial 4 finished with value: -13725.6

## 3. Evaluation & Plots
Plot the compute trials (iterations) vs. cross-validation error, and plot the hyperparameter space to visualize how the Bayesian method explored the space.

In [4]:
# TODO: Generate and save trials_vs_error.png
# X-axis: Number of iterations
# Y-axis: Best CV error found so far
# Overlay Grid, Random, and Bayesian methods on the same plot.
grid_mae = -np.maximum.accumulate(grid_search.cv_results_['mean_test_score'])
random_mae = -np.maximum.accumulate(random_search.cv_results_['mean_test_score'])
bayes_trials = np.array([trial.value for trial in study.trials if trial.value is not None], dtype=float)
bayes_mae = -np.maximum.accumulate(bayes_trials)

plt.figure(figsize=(10, 6))
plt.plot(range(1, len(grid_mae) + 1), grid_mae, label='Grid Search')
plt.plot(range(1, len(random_mae) + 1), random_mae, label='Random Search')
plt.plot(range(1, len(bayes_mae) + 1), bayes_mae, label='Bayesian Optimization')
plt.xlabel('Number of Iterations')
plt.ylabel('Best CV MAE')
plt.title('Hyperparameter Tuning Methods Comparison')
plt.legend()
plt.tight_layout()
plt.savefig('./plots/trials_vs_error.png')
plt.close()

# TODO: Generate and save optuna_hyperparameter_space.png
ax = optuna.visualization.matplotlib.plot_optimization_history(study)
plt.tight_layout()
plt.savefig('./plots/optuna_hyperparameter_space.png')
plt.close()

C:\Users\ratho\AppData\Local\Temp\ipykernel_22356\2733964380.py:23: ExperimentalWarning: optuna.visualization.matplotlib._optimization_history.plot_optimization_history is experimental (supported from v2.2.0). The interface can change in the future.
  ax = optuna.visualization.matplotlib.plot_optimization_history(study)


## 4. Final Testing & Model Saving
Report the best hyperparameters found, train your overall best model on the entire `train.csv`, evaluate on `test.csv`, and save the model file.

In [ ]:
# Print the best hyperparameters found by all 3 methods
grid = {"param": grid_search.best_params_, "mae": grid_best_mae}
random = {"param": random_search.best_params_, "mae": random_best_mae}
bayes = {"param": study.best_params, "mae": bayes_best_mae}

param_df = pd.DataFrame({
    'Method': ['Grid Search', 'Random Search', 'Bayesian Optimization'],
    'Best MAE': [grid['mae'], random['mae'], bayes['mae']],
    'n_estimators': [grid['param']['n_estimators'], random['param']['n_estimators'], bayes['param']['n_estimators']],
    'max_depth': [grid['param']['max_depth'], random['param']['max_depth'], bayes['param']['max_depth']],
    'min_samples_split': [grid['param']['min_samples_split'], random['param']['min_samples_split'], bayes['param']['min_samples_split']],
})

print("Best Hyperparameters from Each Method:")
print(param_df)

best_params = min([grid, random, bayes], key=lambda item: item['mae'])['param']

print("Best Hyperparameters Overall:", best_params)

best_model = RandomForestRegressor(**best_params, random_state=42, n_jobs=-1)

# TODO: Train the best model found on the full X_train
best_model.fit(X_train, y_train)

# TODO: Evaluate the model on X_test (Report MAE)
y_pred = best_model.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
print("Mean Absolute Error on Test Set:", mae)

trackio.log({
    'method': 'Final Best Model Evaluation',
    'test_mae': float(mae)
})

trackio.finish()

# TODO: Save best_model.pkl and any necessary encoders to the models/ folder
with open('./models/best_rf_model.pkl', 'wb') as f:
    pickle.dump(best_model, f)
    
with open('./models/location_encoder.pkl', 'wb') as f:
    pickle.dump(location_encoder, f)
with open('./models/city_encoder.pkl', 'wb') as f:
    pickle.dump(city_encoder, f)
with open('./models/status_encoder.pkl', 'wb') as f:
    pickle.dump(status_encoder, f)
with open('./models/property_type_encoder.pkl', 'wb') as f:
    pickle.dump(property_type_encoder, f)

Best Hyperparameters from Each Method:
                  Method      Best MAE  n_estimators  max_depth  \
0            Grid Search  13268.933395           200         25   
1          Random Search  13298.719930           142         24   
2  Bayesian Optimization  13277.303816           152         28   

   min_samples_split  
0                  2  
1                  2  
2                  2  
Best Hyperparameters Overall: {'max_depth': 25, 'min_samples_split': 2, 'n_estimators': 200}
Mean Absolute Error on Test Set: 12417.007287299955
* Run finished. Uploading logs to Trackio (please wait...)


Exception in callback _ProactorBasePipeTransport._call_connection_lost(None)
handle: <Handle _ProactorBasePipeTransport._call_connection_lost(None)>
Traceback (most recent call last):
  File "C:\Program Files\WindowsApps\PythonSoftwareFoundation.Python.3.12_3.12.2800.0_x64__qbz5n2kfra8p0\Lib\asyncio\events.py", line 88, in _run
    self._context.run(self._callback, *self._args)
  File "C:\Program Files\WindowsApps\PythonSoftwareFoundation.Python.3.12_3.12.2800.0_x64__qbz5n2kfra8p0\Lib\asyncio\proactor_events.py", line 165, in _call_connection_lost
    self._sock.shutdown(socket.SHUT_RDWR)
ConnectionResetError: [WinError 10054] An existing connection was forcibly closed by the remote host
Exception in callback _ProactorBasePipeTransport._call_connection_lost(None)
handle: <Handle _ProactorBasePipeTransport._call_connection_lost(None)>
Traceback (most recent call last):
  File "C:\Program Files\WindowsApps\PythonSoftwareFoundation.Python.3.12_3.12.2800.0_x64__qbz5n2kfra8p0\Lib\asyncio\ev